In [ ]:
import numpy as np

class GridWorldMDP:
    def __init__(self,size,goal,trap):
        self.size=size
        self.goal=goal
        self.trap=trap
        self.state_space=[(i,j) for i in range(size) for j in range(size)]
        self.action_space=['UP','DOWN','LEFT','RIGHT']
        self.transitions=self.build_transitions()
        self.rewards=self.build_rewards()

    def build_transitions(self):
        transitions={}
        for state in self.state_space:
            transitions[state]={}
            for action in self.action_space:
                transitions[state][action]=self.calculate_transitions(state,action)
        return transitions

    def calculate_transitions(self,state,action):
        i,j = state
        if action == 'UP':
            return self.validate_state(i-1,j)
        elif action == 'DOWN':
            return self.validate_state(i+1,j)
        elif action == 'LEFT':
            return self.validate_state(i,j-1)
        elif action == 'RIGHT':
            return self.validate_state(i,j+1)

    def validate_state(self,i,j):
        i = max(0,min(i,self.size-1))
        j = max(0,min(j,self.size-1))
        return [(1.0,(i,j))]

    def build_rewards(self):
        rewards={}
        for state in self.state_space:
            rewards[state] = -1.0
        rewards[self.goal] = 0.0
        rewards[self.trap] = -10.0
        return rewards

size = 3
goal = (2,2)
trap = (1,1)
mdp = GridWorldMDP(size,goal,trap)

In [ ]:
def value_iteration(mdp,gamma=0.9,epsilon=0.01):
    state_values={state:0.0 for state in mdp.state_space}
    ctr = 0
    while True:
        delta = 0
        for state in mdp.state_space:
            if state==mdp.goal or state==mdp.trap:
                    continue
            v = state_values[state]
            state_values[state] = max([sum([p*(mdp.rewards[next_state]+gamma*state_values[next_state])
                              for p,next_state in mdp.transitions[state][action]]) for action in mdp.action_space])
            delta = max(delta,abs(v-state_values[state]))
        print(f"\nIteration {ctr} : ")
        for state,value in state_values.items():
            print(f"State:{state}, Value:{value}")
        ctr += 1
        if delta < epsilon:
            break
    return state_values

print("Value Iteration")
value_iteration_result=value_iteration(mdp)
for state,value in value_iteration_result.items():
    print(f"State:{state}, Value:{value}")

Value Iteration

Iteration 0 : 
State:(0, 0), Value:-1.0
State:(0, 1), Value:-1.0
State:(0, 2), Value:-1.0
State:(1, 0), Value:-1.0
State:(1, 1), Value:0.0
State:(1, 2), Value:0.0
State:(2, 0), Value:-1.0
State:(2, 1), Value:0.0
State:(2, 2), Value:0.0

Iteration 1 : 
State:(0, 0), Value:-1.9
State:(0, 1), Value:-1.9
State:(0, 2), Value:-1.0
State:(1, 0), Value:-1.9
State:(1, 1), Value:0.0
State:(1, 2), Value:0.0
State:(2, 0), Value:-1.0
State:(2, 1), Value:0.0
State:(2, 2), Value:0.0

Iteration 2 : 
State:(0, 0), Value:-2.71
State:(0, 1), Value:-1.9
State:(0, 2), Value:-1.0
State:(1, 0), Value:-1.9
State:(1, 1), Value:0.0
State:(1, 2), Value:0.0
State:(2, 0), Value:-1.0
State:(2, 1), Value:0.0
State:(2, 2), Value:0.0

Iteration 3 : 
State:(0, 0), Value:-2.71
State:(0, 1), Value:-1.9
State:(0, 2), Value:-1.0
State:(1, 0), Value:-1.9
State:(1, 1), Value:0.0
State:(1, 2), Value:0.0
State:(2, 0), Value:-1.0
State:(2, 1), Value:0.0
State:(2, 2), Value:0.0
State:(0, 0), Value:-2.71
State:(0

In [ ]:
def policy_iteration(mdp,gamma=0.9):
    policy = {
        state:np.random.choice(mdp.action_space)
        for state in mdp.state_space if state != mdp.goal and state != mdp.trap
    }
    state_values = {state:0.0 for state in mdp.state_space}
    ctr=0
    while True:
        while True:
            delta = 0
            for state in mdp.state_space:
                if state == mdp.goal or state == mdp.trap:
                    continue
                v = state_values[state]
                action=policy[state]
                state_values[state] = sum([p*(mdp.rewards[next_state]+gamma*state_values[next_state])
                                       for p,next_state in mdp.transitions[state][action]])
                delta = max(delta,abs(v-state_values[state]))
            if delta < 0.01:
                break
        policy_stable=True
        for state in mdp.state_space:
            if state==mdp.goal or state==mdp.trap:
                continue
            old_action=policy[state]
            policy[state]=max(mdp.action_space,key=lambda a:
                            sum([p*(mdp.rewards[next_state]+gamma*state_values[next_state])
                           for p,next_state in mdp.transitions[state][a]]))
            if old_action != policy[state]:
                policy_stable=False
        print(f"\nIteration {ctr} : ")
        for state,action in policy.items():
            print(f"State:{state}, Action:{action}, Value:{state_values[state]}")
        ctr += 1
        if policy_stable:
              break
    return policy,state_values

print("Policy Iteration")
policy_iteration_result,policy_iteration_state_values=policy_iteration(mdp)
for state,action in policy_iteration_result.items():
    print(f"State:{state}, Action:{action}, Value:{policy_iteration_state_values[state]}")

Policy Iteration

Iteration 0 : 
State:(0, 0), Action:UP, Value:-9.912720364319124
State:(0, 1), Action:LEFT, Value:-9.921448327887212
State:(0, 2), Action:UP, Value:-9.912720364319124
State:(1, 0), Action:UP, Value:-10.0
State:(1, 2), Action:DOWN, Value:-9.912720364319124
State:(2, 0), Action:RIGHT, Value:-9.912720364319124
State:(2, 1), Action:RIGHT, Value:0.0

Iteration 1 : 
State:(0, 0), Action:UP, Value:-9.929303495098491
State:(0, 1), Action:LEFT, Value:-9.936373145588643
State:(0, 2), Action:DOWN, Value:-9.929303495098491
State:(1, 0), Action:DOWN, Value:-9.936373145588643
State:(1, 2), Action:DOWN, Value:0.0
State:(2, 0), Action:RIGHT, Value:-1.0
State:(2, 1), Action:RIGHT, Value:0.0

Iteration 2 : 
State:(0, 0), Action:DOWN, Value:-9.942735831029779
State:(0, 1), Action:RIGHT, Value:-9.948462247926802
State:(0, 2), Action:DOWN, Value:-1.0
State:(1, 0), Action:DOWN, Value:-1.9
State:(1, 2), Action:DOWN, Value:0.0
State:(2, 0), Action:RIGHT, Value:-1.0
State:(2, 1), Action:RIGHT

In [31]:
# Value Iteration for traffic signals
states = ["Low", "Medium", "High"]
actions = ["Short", "Medium", "Long"]
gamma = 0.9

# Reward based on traffic density
reward = {"Low": 10, "Medium": 5, "High": -10}
V = {s:0 for s in states}

# Iteratively update state values
for _ in range(20):
    for s in states:
        V[s] = max(reward[s] + gamma * V[s] for a in actions)

print("Optimal State Values for Traffic Control:", V)


Optimal State Values for Traffic Control: {'Low': 87.84233454094309, 'Medium': 43.92116727047154, 'High': -87.84233454094309}


In [29]:
# Policy Iteration for warehouse robot
states = ["Idle", "Working"]
actions = ["Work", "Charge"]
gamma = 0.9

# Initialize policy & state values
policy = {s: "Work" for s in states}
V = {s: 0 for s in states}

# Policy iteration loop
for _ in range(10):
    # Policy evaluation
    for s in states:
        V[s] = 5 + gamma * V[s]
    # Policy improvement
    for s in states:
        policy[s] = "Work" if V[s] > 2 else "Charge"
print("Optimal Warehouse Robot Policy:", policy)


Optimal Warehouse Robot Policy: {'Idle': 'Work', 'Working': 'Work'}
